# Train every model on a Colab GPU

This machine is CPU-only, so a full sweep takes hours. On a Colab T4 the same
sweep is minutes. Run the cells top to bottom.

**First: Runtime -> Change runtime type -> T4 GPU.** Without that this is no
faster than a laptop.

Nothing here is committed back to GitHub — the last cell downloads the trained
weights and model cards so you can drop them into `model/` locally.


## 1. GPU check
Stop here if this prints nothing — you are on a CPU runtime.


In [ ]:
!nvidia-smi -L


## 2. Clone the repo and install deps

Colab already ships tensorflow, numpy, sklearn, pandas, pyarrow and pillow.
Only `wfdb` (MIT-BIH download) and `kagglehub` (brain MRI dataset) are missing.


In [ ]:
!git clone https://github.com/AlphaSukehero/ai-detection.git
%cd ai-detection
!pip install -q wfdb kagglehub


## 3. Fetch the raw data

None of the datasets are in git — they are all reproducible from scripts.

- **EuroSAT** and **MIT-BIH** download themselves via `scripts/download_raw.py`.
- **Brain MRI** comes from the Kaggle tree the repo expects at `dataset/`;
  `kagglehub` pulls it without needing an API token.

This cell is the slow one (a few minutes, mostly the 44 MIT-BIH records).


In [ ]:
import os, kagglehub

# EuroSAT parquet shards + MIT-BIH records -> data/raw/
!python scripts/download_raw.py all

# Brain MRI -> dataset/{Training,Testing}, which prepare_mri.py reads.
src = kagglehub.dataset_download('masoudnickparvar/brain-tumor-mri-dataset')
os.makedirs('dataset', exist_ok=True)
for split in ('Training', 'Testing'):
    dst = f'dataset/{split}'
    if not os.path.exists(dst):
        os.symlink(f'{src}/{split}', dst)
print(sorted(os.listdir('dataset')))


## 4. Build the processed splits

Deterministic (`SEED = 42`) and idempotent, so re-running is safe. Each script
writes a SHA-256 manifest to `data/manifests/` and asserts no image leaks
across splits.


In [ ]:
!python scripts/prepare_mri.py
!python scripts/prepare_eurosat.py
!python scripts/prepare_ecg.py


## 5. Smoke test before the real run

`EPOCHS=1` proves the whole pipeline is wired up in ~2 minutes. The accuracy
numbers it prints are meaningless — you are checking that all four scripts run
to completion and save a model.


In [ ]:
!EPOCHS=1 scripts/train_all.sh


## 6. Train everything for real

Uses each script's default epoch count (ECG 40, MRI CNN 30, VGG16 60,
EuroSAT 40), all with EarlyStopping. Logs land in `logs/`.

To train a subset instead: `!scripts/train_all.sh ecg eurosat`


In [ ]:
!scripts/train_all.sh


## 7. Check what you got

Every trainer writes a JSON sidecar next to the weights recording the class
order, input shape, preprocessing and test metrics. The class order in the
sidecar is authoritative — inference never re-derives it from a directory
listing.


In [ ]:
import glob, json

for card in sorted(glob.glob('model/*.json')):
    meta = json.load(open(card))
    print(card, json.dumps(meta.get('metrics', meta), indent=2))


## 8. Download the trained models

Pull the archive down and unpack it over your local `model/` directory.
The weights are gitignored, so this is how they get home.


In [ ]:
!tar czf models.tar.gz model/ logs/
from google.colab import files
files.download('models.tar.gz')


---

Unpack locally with:

```bash
tar xzf ~/Downloads/models.tar.gz -C /home/alpha/Desktop/ai_detection
```
